# FINAL RECOVERY — one Colab, then decide TEST3

```text
B1 Linux E2T → B1 cache replay → B2 Agent.solve Tables → B2b K (optional)
→ B3 No-files → B4 static → BAKE → RECOVERY_READY
```

**STOP** at first failed gate. No Qwen3 / new prompts / rules / auto-submit.

If `RECOVERY_READY`: submit `STAGE3_E2T_RECOVERY.zip` manually as TEST3.


In [14]:
!git clone https://github.com/BeediGoua/TEST-PROJET-ML.git
!cd /content/TEST-PROJET-ML && git pull
from IPython.display import display, HTML

file_path = "/content/TEST-PROJET-ML/audit_independent/colab_final_recovery.ipynb"
print(f"Fichier : {file_path}")
print("Fichiers (gauche) → TEST-PROJET-ML → audit_independent → colab_final_recovery.ipynb")



Cloning into 'TEST-PROJET-ML'...
remote: Enumerating objects: 321, done.
remote: Counting objects: 100% (321/321), done.
remote: Compressing objects: 100% (267/267), done.
remote: Total 321 (delta 60), reused 299 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (321/321), 1.30 MiB | 3.81 MiB/s, done.
Resolving deltas: 100% (60/60), done.
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 31 (delta 18), reused 23 (delta 11), pack-reused 0 (from 0)
Unpacking objects: 100% (31/31), 73.11 KiB | 542.00 KiB/s, done.
From https://github.com/BeediGoua/TEST-PROJET-ML
   7251a2e..2311b55  main       -> origin/main
Updating 7251a2e..2311b55
error: Your local changes to the following files would be overwritten by merge:
	audit_independent/STAGE3_E2T_RECOVERY.zip
Please commit your changes or stash them before you merge.
error: The following untracked working tree files would be overwritten by

## 00 — Setup

In [15]:

from pathlib import Path
import sys, os, platform, json, tempfile, time
assert platform.system()=="Linux", "FINAL RECOVERY requires Colab Linux"
# !git clone https://github.com/BeediGoua/TEST-PROJET-ML.git
PROJECT_ROOT = Path("/content/TEST-PROJET-ML")
assert (PROJECT_ROOT/"audit_independent"/"audit.py").exists()
AUDIT = PROJECT_ROOT/"audit_independent"
sys.path[:0] = [str(AUDIT), str(PROJECT_ROOT), str(AUDIT/"diagnostic")]
os.chdir(AUDIT)
!pip -q install torch transformers accelerate pandas numpy scipy scikit-learn sympy pyarrow 2>/dev/null || true
from gpu_experiments import official
from build_experimental import build_test1_e2t, build_test1_alias
from pretest3_common import EXP, load_path, AUDIT_OUT, DIAG_OUT, table_tasks, load_cache, is_real_qwen_cache
import audit as audit_mod
from src.evaluation.core import _materialize_all
import final_recovery_harness as H
from bake_stage3_e2t_recovery import bake
build_test1_alias(); build_test1_e2t(); official()
e2t = load_path(EXP/"test1_e2t_dsh.py", "e2t")
stage3 = audit_mod.load("stage3")
CACHE = AUDIT_OUT/"generations_34d6ece55b3c092a.json"
assert CACHE.exists(), "Upload frozen cache into audit_independent/results/"
cache = load_cache(CACHE); assert is_real_qwen_cache(cache)
print("OK setup", platform.platform())


OK setup Linux-6.6.122+-x86_64-with-glibc2.39


## B1 — Linux E2T contract (STOP if fail)

In [16]:

tasks = table_tasks(); t0 = tasks[0]
with tempfile.TemporaryDirectory() as td:
    fm = _materialize_all([t0], td)
    H.b1_linux_contract(e2t, stage3, t0, fm)


PASS B1_linux_contract {'loop': '[6.0, None, 42.0, 9.0]'}


## B1 — Cache replay 400 programs (task-level gate)

In [17]:

OUTER = float(os.environ.get("DSH_OUTER_MANY", "7.5"))
gold = {t["id"]: t for t in table_tasks()}
with tempfile.TemporaryDirectory() as td:
    fm = _materialize_all(list(gold.values()), td)
    m = H.b1_cache_replay(e2t, stage3, cache, gold, fm, outer=OUTER)
print(m)


PASS B1_cache_replay {'program_agree': 1.0, 'task_vote_agree': 1.0, 'stage3_accuracy': 0.7875, 'e2t_accuracy': 0.7875, 'none_where_e0_finite': 0, 'bad_numeric': 0, 'p50': 1.0997013940000215, 'p95': 1.3269960600000559, 'max': 2.6845091010000033, 'outer': 7.5}
{'program_agree': 1.0, 'task_vote_agree': 1.0, 'stage3_accuracy': 0.7875, 'e2t_accuracy': 0.7875, 'none_where_e0_finite': 0, 'bad_numeric': 0, 'p50': 1.0997013940000215, 'p95': 1.3269960600000559, 'max': 2.6845091010000033, 'outer': 7.5}


## B2 — REAL Agent.solve Tables (forced fallback) — GATE PRINCIPAL

In [25]:
import os, sys, gc, tempfile
from pathlib import Path
import torch

PROJECT_ROOT = Path("/content/TEST-PROJET-ML")
AUDIT = PROJECT_ROOT / "audit_independent"
sys.path[:0] = [str(AUDIT), str(PROJECT_ROOT), str(AUDIT / "diagnostic")]
os.chdir(AUDIT)

# Libérer la VRAM (session précédente)
for name in ["agent_a", "agent_b", "A", "Bmod"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
print("GPU free MiB:", torch.cuda.mem_get_info()[0] // 1024**2)

import final_pre_test3_debug as D
from bake_stage3_e2t_recovery import bake
from pretest3_common import table_tasks
from src.evaluation.core import _materialize_all
import final_recovery_harness as H
from gpu_experiments import official

official()
OUTER = 7.5
os.environ["DSH_FORCE_FALLBACK"] = "1"
os.environ["DSH_OUTER_MANY"] = str(OUTER)

agent_path = bake(outer_many=OUTER)
Bmod = H.load_mod(agent_path, "stage3_e2t_rec")
Bmod.MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
agent_b = Bmod.Agent()  # UN seul LLM

tables = table_tasks()
TD = tempfile.mkdtemp(prefix="b2val_")
fm = _materialize_all(tables, TD)
first = [{**t, "files": fm[t["id"]]} for t in tables[:8]]

st_b = D.instrumented_solve(agent_b, Bmod, first, "E2T")
print("E2T", {k: st_b[k] for k in st_b if k != "answers"})

assert st_b["deterministic_count"] == 0 and st_b["llm_live_count"] == 8
assert st_b["greedy_generation_seconds"] > 1
print("B2 instrument PASS — vrai LLM forced-fallback")

GPU free MiB: 3131


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[init] modèle Qwen/Qwen2.5-Coder-1.5B-Instruct sur cuda:0, échantillons 4, budget 29.0s
[solve] prep 0.3s | glouton 19.9s | réparation 8.0s | échantillons 0.0s | total 28.2s
E2T {'label': 'E2T', 'n_tasks': 8, 'deterministic_count': 0, 'llm_live_count': 8, 'greedy_generation_seconds': 10.805612459000258, 'greedy_outputs_count': 8, 'repair_count': 3, 'sample_outputs_count': 0, 'total_seconds': 28.156740888999593}
B2 instrument PASS — vrai LLM forced-fallback


## B2b — workload K (optional, cheap stress)

In [19]:

# Quantify overhead only; does not veto unless a batch >40s (harness already STOPS)
Ks = [1, 2, 4, 8]
stress = []
# use first 8 tables; mark K as fallback by FORCE_FALLBACK already on all — simulate K by solving subsets
for K in Ks:
    sub = batch_tasks[:K]
    # pad with deterministic answered dummies? skip — just time K-task solve
    t0 = time.perf_counter(); agent_b.solve(sub); dt = time.perf_counter()-t0
    stress.append(dict(K=K, seconds=dt))
    print("K", K, "s", dt)
    if dt > 40:
        H.stop(f"B2b K={K} >40s")
import pandas as pd
pd.DataFrame(stress).to_csv(DIAG_OUT/"final_b2b_stress.csv", index=False)
H.mark("B2b_stress", **{f"K{s['K']}": s["seconds"] for s in stress})


[solve] prep 0.0s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.0s
K 1 s 0.03011367799990694
[solve] prep 0.1s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.1s
K 2 s 0.07081292599968947
[solve] prep 0.2s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.2s
K 4 s 0.15495402900023691
[solve] prep 0.3s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.3s
K 8 s 0.2838166189999356
PASS B2b_stress {'K1': 0.03011367799990694, 'K2': 0.07081292599968947, 'K4': 0.15495402900023691, 'K8': 0.2838166189999356}


## B3 — No-files uncovered by calc_rules

In [20]:
import math, time, os
import pandas as pd

# Rebuild nofiles if needed
HANDOFF = PROJECT_ROOT / "CODEX_AUDIT_HANDOFF"
all_tasks = json.loads((HANDOFF / "dev.json").read_text(encoding="utf-8"))
nofiles = []
for t in all_tasks:
    if t.get("files"):
        continue
    try:
        rv = A.calc_rules(t["objective"])
    except Exception:
        rv = None
    if rv is None or not (isinstance(rv, (int, float)) and math.isfinite(float(rv))):
        nofiles.append(t)

TD_NF = tempfile.mkdtemp(prefix="final_nofiles_")
fm = _materialize_all(nofiles, TD_NF)
nf_tasks = [{**t, "files": fm[t["id"]]} for t in nofiles]

os.environ.pop("DSH_FORCE_FALLBACK", None)
rows = []
for start in range(0, len(nf_tasks), 8):
    batch = nf_tasks[start:start+8]
    t0 = time.perf_counter(); ra = agent_a.solve(batch); ta = time.perf_counter() - t0
    t0 = time.perf_counter(); rb = agent_b.solve(batch); tb = time.perf_counter() - t0
    by_a = {r["id"]: r["answer"] for r in ra}
    by_b = {r["id"]: r["answer"] for r in rb}
    for t in batch:
        ans, tol = float(t["answer"]), float(t["tol"])
        va, vb = by_a.get(t["id"]), by_b.get(t["id"])
        rows.append(dict(
            id=t["id"], gold=ans, a=va, b=vb,
            a_ok=va is not None and abs(float(va)-ans) <= tol,
            b_ok=vb is not None and abs(float(vb)-ans) <= tol,
            time_a=ta, time_b=tb,
        ))

df = pd.DataFrame(rows)
path = DIAG_OUT / "final_b3_nofiles.csv"
df.to_csv(path, index=False)
print("saved", path)
print("Stage3", df.a_ok.mean(), "E2T", df.b_ok.mean())
lost = df[(df.a_ok == True) & (df.b_ok == False)]
print("Stage3 OK, E2T KO:", len(lost))
print(lost[["id", "gold", "a", "b"]].to_string(index=False))

[solve] prep 0.0s | glouton 5.4s | réparation 7.4s | échantillons 10.0s | total 22.8s
[solve] prep 0.0s | glouton 16.5s | réparation 6.9s | échantillons 0.0s | total 23.4s
[solve] prep 0.0s | glouton 5.1s | réparation 5.8s | échantillons 8.8s | total 19.7s
[solve] prep 0.0s | glouton 16.1s | réparation 9.9s | échantillons 0.0s | total 26.0s
[solve] prep 0.0s | glouton 3.1s | réparation 0.0s | échantillons 6.3s | total 9.4s
[solve] prep 0.0s | glouton 14.1s | réparation 0.0s | échantillons 14.3s | total 28.4s
[solve] prep 0.0s | glouton 4.2s | réparation 4.8s | échantillons 8.3s | total 17.4s
[solve] prep 0.0s | glouton 15.4s | réparation 7.6s | échantillons 5.6s | total 28.5s
[solve] prep 0.0s | glouton 4.6s | réparation 0.0s | échantillons 8.4s | total 12.9s
[solve] prep 0.0s | glouton 15.8s | réparation 0.0s | échantillons 12.1s | total 27.9s
saved /content/TEST-PROJET-ML/audit_independent/diagnostic/results/final_b3_nofiles.csv
Stage3 0.975 E2T 0.9
Stage3 OK, E2T KO: 4
   id  gold  

## B4 — Static / resource check + BAKE decision

In [21]:

rep = H.b4_static_and_resources(agent_path, AUDIT/"STAGE3_E2T_RECOVERY"/"dsh.py")
# Re-bake fresh zip after all greens
bake(outer_many=OUTER)
H.finalize_ready()
print("Download STAGE3_E2T_RECOVERY.zip and submit TEST3 manually if you accept RECOVERY_READY.")


PASS B4_static {'agent_file': 'agent.py', 'exec_upload_violation': False, 'sympy_prompt_runtime_mismatch': True, 'onehot_prompt_runtime_mismatch': True, 'no_imports_example_mismatch': True, 'prompt_runtime_mismatch_count': 3, 'zip_bytes': 44043, 'nfiles': 2}
# FINAL RECOVERY

**RECOVERY_READY**

Gates passed:
- B1_linux_contract: {'loop': '[6.0, None, 42.0, 9.0]'}
- B1_cache_replay: {'program_agree': 1.0, 'task_vote_agree': 1.0, 'stage3_accuracy': 0.7875, 'e2t_accuracy': 0.7875, 'none_where_e0_finite': 0, 'bad_numeric': 0, 'p50': 1.0997013940000215, 'p95': 1.3269960600000559, 'max': 2.6845091010000033, 'outer': 7.5}
- solve_E2T_vs_Stage3: {'n': 80, 'acc_a': 1.0, 'acc_b': 1.0, 'task_agree': 1.0, 'max_batch_a': 0.34328226099978565, 'max_batch_b': 0.3551596580000478}
- B2b_stress: {'K1': 0.03011367799990694, 'K2': 0.07081292599968947, 'K4': 0.15495402900023691, 'K8': 0.2838166189999356}
- B4_static: {'agent_file': 'agent.py', 'exec_upload_violation': False, 'sympy_prompt_runtime_mismatch'

In [23]:
!cd /content/TEST-PROJET-ML && git fetch origin && git reset --hard origin/main && git log -1 --oneline

from pathlib import Path
p = Path("/content/TEST-PROJET-ML/audit_independent/final_pre_test3_debug.py")
print("exists:", p.exists(), p)

HEAD is now at 2311b55 Mark prior B2 INVALID; dynamic FORCE_FALLBACK; pre-TEST3 debug harness.
2311b55 (HEAD -> main, origin/main, origin/HEAD) Mark prior B2 INVALID; dynamic FORCE_FALLBACK; pre-TEST3 debug harness.
exists: True /content/TEST-PROJET-ML/audit_independent/final_pre_test3_debug.py


In [22]:
import os, sys, tempfile
from pathlib import Path

PROJECT_ROOT = Path("/content/TEST-PROJET-ML")
AUDIT = PROJECT_ROOT / "audit_independent"
sys.path[:0] = [str(AUDIT), str(PROJECT_ROOT), str(AUDIT / "diagnostic")]
os.chdir(AUDIT)

import final_pre_test3_debug as D
from bake_stage3_e2t_recovery import bake
from pretest3_common import table_tasks
from src.evaluation.core import _materialize_all
import audit as audit_mod
import final_recovery_harness as H
from gpu_experiments import official

official()
OUTER = 7.5
os.environ["DSH_FORCE_FALLBACK"] = "1"
os.environ["DSH_OUTER_MANY"] = str(OUTER)

agent_path = bake(outer_many=OUTER)
A = audit_mod.load("stage3")
D.disable_table_rules(A)
A._rules_disabled = True
Bmod = H.load_mod(agent_path, "stage3_e2t_rec")
A.MODEL = Bmod.MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
agent_a, agent_b = A.Agent(), Bmod.Agent()

tables = table_tasks()
TD = tempfile.mkdtemp(prefix="b2val_")
fm = _materialize_all(tables, TD)
first = [{**t, "files": fm[t["id"]]} for t in tables[:8]]

st_a = D.instrumented_solve(agent_a, A, first, "Stage3")
st_b = D.instrumented_solve(agent_b, Bmod, first, "E2T")
for st in (st_a, st_b):
    print(st["label"], {k: st[k] for k in st if k != "answers"})

assert st_a["deterministic_count"] == 0 and st_a["llm_live_count"] == 8
assert st_b["deterministic_count"] == 0 and st_b["llm_live_count"] == 8
assert st_a["greedy_generation_seconds"] > 1 and st_b["greedy_generation_seconds"] > 1
print("B2 instrument PASS")

ModuleNotFoundError: No module named 'final_pre_test3_debug'

In [26]:
import pandas as pd
from pretest3_common import DIAG_OUT

csv = DIAG_OUT / "final_b3_nofiles.csv"
df = pd.read_csv(csv)
lost = df[(df.a_ok == True) & (df.b_ok == False)]
gain = df[(df.a_ok == False) & (df.b_ok == True)]
print("LOST", len(lost)); print(lost[["id","gold","a","b"]].to_string(index=False))
print("GAIN", len(gain)); print(gain[["id","gold","a","b"]].to_string(index=False))

LOST 4
   id  gold    a   b
d_026  13.0 13.0 0.0
d_065   6.0  6.0 0.0
d_076  16.0 16.0 0.0
d_143   8.9  8.9 0.0
GAIN 1
   id  gold    a    b
d_123  27.0 13.0 27.0
